# Falcon 9 Web Scraping

**Author:** Aung Paing Soe


## Request page

Use requests and BeautifulSoup to collect the Falcon 9 launch tables from Wikipedia.


In [ ]:
import requests, pandas as pd, numpy as np
from bs4 import BeautifulSoup
url='https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches'
soup=BeautifulSoup(requests.get(url,timeout=30).text,'html.parser')


## Parse tables

Read header and data cells, remove citation whitespace, retain rows beginning with a flight number, and de-duplicate rows.


In [ ]:
records=[]
for table in soup.select('table.wikitable'):
    for row in table.select('tr'):
        cells=[' '.join(c.get_text(' ',strip=True).split()) for c in row.select('th,td')]
        if cells and cells[0].split()[0].isdigit(): records.append(cells)
scraped=pd.DataFrame(records).drop_duplicates()
scraped.head()


## Clean fields

Standardize launch date, booster, payload mass, orbit, customer, launch site, and outcome before joining with API records.


In [ ]:
def clean_mass(v):
    token=''.join(c for c in v.split('kg')[0] if c.isdigit() or c=='.')
    return float(token) if token else np.nan
print('Parsed rows:',len(scraped))
